# E-PATH-CO-REASON: Production-Grade Google Colab Training Notebook

This notebook serves as the **unified, production-grade training and evaluation pipeline** for the E-PATH-CO-REASON research experiments. It is designed to run end-to-end on a completely fresh Google Colab GPU runtime (or a local environment) with a single "Run All" command.

## 1. Notebook Purpose
This notebook orchestrates the training, validation, evaluation, and logging of the **E-PATH-CO-REASON** model. It includes mechanisms for differentiable path routing via Gumbel-Softmax, representation alignment via Dynamic Consistency Projection (DCP), and multi-objective composite loss tracking.

## 2. Directory Layout & Persistence
All outputs are persistently saved inside the experiments workspace folder:
```
experiments/<experiment_name>/
├── best_model.pt             # Best model checkpoint
├── latest_model.pt           # Last completed epoch checkpoint
├── checkpoints/              # Checkpoint directory
├── logs/                     # environment.json and configuration.json
├── exports/                  # training_history.csv, metrics.json, routing_statistics.json
└── figures/                  # loss_curves.png, accuracy_curves.png, confusion matrices
```

## 3. Running & Resuming
- **First Run**: Select `Runtime -> Run All`. The notebook will automatically check imports, setup the repository, validate the dataset, and start training from scratch.
- **Resuming Interrupted Runs**: If training gets disconnected, select `Runtime -> Run All` again. The notebook automatically mounts Google Drive, detects the existing `latest_model.pt` checkpoint, and resumes training from the exact interrupted epoch/optimizer/scheduler/seed state.

## 1. Central Experiment Configuration Block

In [ ]:
# ==========================================
# CENTRAL EXPERIMENT CONFIGURATION BLOCK
# ==========================================
EXPERIMENT_CONFIG = {
    "experiment_name": "epath_co_reason_baseline",
    "git_branch": "main",  # Branch to clone/checkout if not already local
    
    # Dataset Parameters
    "dataset_relative_path": "meditriage/data/processed/dataset.csv",
    "max_samples": None,  # Set to an integer (e.g. 500) to train on subset, or None for the full dataset
    "max_length": 128,
    
    # Split Parameters
    "train_ratio": 0.8,
    "val_ratio": 0.1,
    "test_ratio": 0.1,
    
    # Trainer Parameters
    "epochs": 10,
    "batch_size": 32,
    "learning_rate": 1e-4,
    "encoder_lr": 2e-5,
    "weight_decay": 0.01,
    "gradient_clipping": 1.0,
    "gradient_accumulation_steps": 1,
    "use_amp": True,
    "seed": 1337,
    "optimizer_type": "adamw",
    "scheduler_type": "cosine",
    "warmup_ratio": 0.1,
    
    # Early Stopping
    "early_stopping_patience": 3,
    "early_stopping_metric": "val_loss",
    "early_stopping_min_improvement": 1e-4,
    
    # Storage and Environment
    "use_drive": True,
    "drive_workspace_dir": "/content/drive/MyDrive/MediTriageAI"
}

## 2. Automatic Repository Setup & Import Validation

In [ ]:
# ==========================================
# SECTION 1 & 3: REPOSITORY DETECT & IMPORT CHECKS
# ==========================================
import os
import sys
from pathlib import Path

# Walk up parent tree to find repo root
def find_repo_root(start_dir: Path) -> Path:
    for parent in [start_dir] + list(start_dir.parents):
        if (parent / ".git").exists() or (parent / "requirements.txt").exists():
            return parent
    return start_dir

curr_dir = Path(os.getcwd()).resolve()
repo_root = find_repo_root(curr_dir)
print(f"Repository Root Detected: {repo_root}")

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

# Clone check (for fresh Colab runtimes)
if not (repo_root / "requirements.txt").exists():
    REPO_URL = "https://github.com/ROHAN-BHUTANI/MediTriageAI.git"
    REPO_DIR = "MediTriageAI_Data_Engine"
    print(f"Cloning fresh repository: {REPO_URL}...")
    !git clone -b {EXPERIMENT_CONFIG['git_branch']} {REPO_URL}
    repo_root = Path(os.getcwd()).resolve() / REPO_DIR
    if str(repo_root) not in sys.path:
        sys.path.insert(0, str(repo_root))

# Install dependencies
req_path = repo_root / "requirements.txt"
if req_path.exists():
    print(f"Installing dependencies from: {req_path}")
    !pip install -q -r {req_path}
else:
    print("Warning: requirements.txt not found. Performing fallback install...")
    !pip install -q transformers scikit-learn matplotlib seaborn pandas numpy torch psutil

# Verify Imports
print("Validating imports and workspace modules...")
required_imports = [
    ("torch", "torch"),
    ("transformers", "transformers"),
    ("pandas", "pandas"),
    ("numpy", "numpy"),
    ("sklearn", "scikit-learn"),
    ("matplotlib", "matplotlib"),
    ("seaborn", "seaborn"),
    ("psutil", "psutil"),
    ("models.emergent_path_triage.model", "E-PATH-CO-REASON Model"),
    ("src.data_pipeline", "E-PATH-CO-REASON Data Pipeline"),
    ("src.trainer", "E-PATH-CO-REASON Trainer")
]

missing = []
for mod_name, friendly_name in required_imports:
    try:
        __import__(mod_name)
        print(f"  ✓ {mod_name} imported successfully.")
    except ImportError as e:
        print(f"  ✗ Failed to import {mod_name}: {e}")
        missing.append(friendly_name)

if missing:
    raise ImportError(f"Verification failed. Missing required components: {missing}")
print("All imports validated successfully.")

## 3. Environment Validation

In [ ]:
# ==========================================
# SECTION 2: ENVIRONMENT VALIDATION
# ==========================================
import json
import torch
import psutil
import transformers
from src.data_pipeline import detect_colab_environment

env_meta = detect_colab_environment()
has_gpu = env_meta["has_gpu"]
gpu_name = env_meta["gpu_name"]
total_vram = 0
free_vram = 0

if has_gpu:
    t = torch.cuda.get_device_properties(0).total_memory
    a = torch.cuda.memory_allocated(0)
    total_vram = t / (1024 ** 3)
    free_vram = (t - a) / (1024 ** 3)

git_commit = "N/A"
try:
    import subprocess
    git_commit = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=str(repo_root)).decode("utf-8").strip()
except Exception:
    pass

env_info = {
    "python_version": sys.version,
    "pytorch_version": torch.__version__,
    "cuda_version": torch.version.cuda if has_gpu else "N/A",
    "transformers_version": transformers.__version__,
    "gpu_model": gpu_name,
    "total_gpu_memory_gb": total_vram,
    "available_gpu_memory_gb": free_vram,
    "cpu_cores": psutil.cpu_count(logical=True),
    "ram_gb": psutil.virtual_memory().total / (1024 ** 3),
    "git_commit_hash": git_commit
}

print("ENVIRONMENT AUDIT LOG:")
print(json.dumps(env_info, indent=4))

## 4. Google Drive Mount & Workspace Setup

In [ ]:
# ==========================================
# SECTION 3: DRIVE MOUNT & FOLDERS VERIFICATION
# ==========================================
if EXPERIMENT_CONFIG["use_drive"] and env_meta["is_colab"]:
    from google.colab import drive
    print("Mounting Google Drive persistently...")
    drive.mount('/content/drive')
    drive_base = Path(EXPERIMENT_CONFIG["drive_workspace_dir"])
else:
    print("Running locally or Drive mount disabled. Saving outputs inside repository root.")
    drive_base = repo_root / "results"

exp_base = drive_base / "experiments" / EXPERIMENT_CONFIG["experiment_name"]

dirs = {
    "experiments": exp_base,
    "checkpoints": exp_base / "checkpoints",
    "logs": exp_base / "logs",
    "figures": exp_base / "figures",
    "exports": exp_base / "exports"
}

for name, path in dirs.items():
    path.mkdir(parents=True, exist_ok=True)
    print(f"Verified folder '{name}': {path}")

# Write permission check
test_file = exp_base / "write_check.txt"
try:
    test_file.write_text("write check successful")
    test_file.unlink()
    print("Write permissions successfully verified.")
except Exception as e:
    raise PermissionError(f"Target folder is not writable: {e}")

# Save environment.json
with open(dirs["logs"] / "environment.json", "w", encoding="utf-8") as f:
    json.dump(env_info, f, indent=4)

## 5. Dataset Validation Checks

In [ ]:
# ==========================================
# SECTION 4: DATASET VALIDATION
# ==========================================
import pandas as pd
from src.data_pipeline import LabelValidator
from transformers import AutoTokenizer

print("Discovering dataset relative to repo root...")
dataset_csv = repo_root / EXPERIMENT_CONFIG["dataset_relative_path"]

if not dataset_csv.exists():
    raise FileNotFoundError(f"Dataset CSV not found at: {dataset_csv}")

df = pd.read_csv(dataset_csv)
print(f"Loaded dataset with {len(df)} total rows.")

# Column validations
required_cols = ["text", "department_code", "severity_heuristic"]
for col in required_cols:
    if col not in df.columns:
        raise KeyError(f"Dataset schema validation failed. Column '{col}' is missing.")

# Drop NaNs in text
nan_text = df["text"].isna().sum()
if nan_text > 0:
    print(f"Removing {nan_text} rows with missing text...")
    df = df.dropna(subset=["text"])

# Mappings validations
validator = LabelValidator()
invalid_spec = (~df["department_code"].isin(validator.specialist_classes)).sum()
invalid_sev = (~df["severity_heuristic"].isin(validator.severity_labels)).sum()

if invalid_spec > 0:
    raise ValueError(f"Invalid specialty labels count: {invalid_spec}")
if invalid_sev > 0:
    raise ValueError(f"Invalid severity labels count: {invalid_sev}")

print("Validating tokenizer encoding format...")
tokenizer = AutoTokenizer.from_pretrained("xlm-roberta-base")
try:
    tokens = tokenizer.encode(df["text"].iloc[0], truncation=True, max_length=EXPERIMENT_CONFIG["max_length"])
    print(f"Tokenizer validation passed. First text encode length: {len(tokens)}")
except Exception as e:
    raise RuntimeError(f"Tokenizer compatibility check failed: {e}")

print("Dataset validation validation PASSED.")

## 6. Training Initialization & Checkpoint Resume Loop

In [ ]:
# ==========================================
# SECTION 6: RESUME OR FIT NEW EXPERIMENT
# ==========================================
import json
from models.emergent_path_triage.model import EmergentPathTriageConfig, EmergentPathTriageModel
from src.data_pipeline import TokenizerPipeline, EmergentTriageDataset, get_dataloader, get_leakage_safe_splits

# Save configuration.json
with open(dirs["logs"] / "configuration.json", "w", encoding="utf-8") as f:
    json.dump(EXPERIMENT_CONFIG, f, indent=4)

print("Preparing dataloaders...")
if EXPERIMENT_CONFIG["max_samples"] is not None:
    df = df.sample(EXPERIMENT_CONFIG["max_samples"], random_state=EXPERIMENT_CONFIG["seed"])

train_df, val_df, test_df = get_leakage_safe_splits(
    df,
    train_ratio=EXPERIMENT_CONFIG["train_ratio"],
    val_ratio=EXPERIMENT_CONFIG["val_ratio"],
    seed=EXPERIMENT_CONFIG["seed"],
    stratify=False
)

pipeline = TokenizerPipeline(tokenizer, max_length=EXPERIMENT_CONFIG["max_length"])

def create_ds(target_df):
    texts = target_df["text"].tolist()
    spec_ids = [validator.validate_specialist(str(c)) for c in target_df["department_code"]]
    sev_ids = [validator.validate_severity(str(l)) for l in target_df["severity_heuristic"]]
    return EmergentTriageDataset(texts, spec_ids, sev_ids, pipeline)

train_loader = get_dataloader(create_ds(train_df), batch_size=EXPERIMENT_CONFIG["batch_size"], shuffle=True)
val_loader = get_dataloader(create_ds(val_df), batch_size=EXPERIMENT_CONFIG["batch_size"], shuffle=False)
test_loader = get_dataloader(create_ds(test_df), batch_size=EXPERIMENT_CONFIG["batch_size"], shuffle=False)

print("Building model...")
config = EmergentPathTriageConfig(latent_dim=8)
model_meta = EmergentPathTriageModel()
model = model_meta.build(None, triage_config=config)

from src.trainer import EmergentTrainer, EmergentTrainerConfig
trainer_cfg = EmergentTrainerConfig(
    epochs=EXPERIMENT_CONFIG["epochs"],
    learning_rate=EXPERIMENT_CONFIG["learning_rate"],
    encoder_lr=EXPERIMENT_CONFIG["encoder_lr"],
    weight_decay=EXPERIMENT_CONFIG["weight_decay"],
    gradient_clipping=EXPERIMENT_CONFIG["gradient_clipping"],
    gradient_accumulation_steps=EXPERIMENT_CONFIG["gradient_accumulation_steps"],
    use_amp=EXPERIMENT_CONFIG["use_amp"],
    early_stopping_patience=EXPERIMENT_CONFIG["early_stopping_patience"],
    early_stopping_metric=EXPERIMENT_CONFIG["early_stopping_metric"],
    early_stopping_min_improvement=EXPERIMENT_CONFIG["early_stopping_min_improvement"],
    warmup_ratio=EXPERIMENT_CONFIG["warmup_ratio"],
    seed=EXPERIMENT_CONFIG["seed"],
    optimizer_type=EXPERIMENT_CONFIG["optimizer_type"],
    scheduler_type=EXPERIMENT_CONFIG["scheduler_type"],
    checkpoint_dir=str(dirs["checkpoints"])
)

trainer = EmergentTrainer(
    model=model,
    config=trainer_cfg,
    train_loader=train_loader,
    val_loader=val_loader,
    tokenizer=tokenizer
)
trainer.checkpoint_dir = dirs["checkpoints"]

# Checkpoint Resume loop check
latest_ckpt = dirs["checkpoints"] / "latest_model.pt"
if latest_ckpt.exists():
    print(f"==================================================")
    print(f"Checkpoint discovered: {latest_ckpt}. Resuming run...")
    print(f"==================================================")
    trainer.load_checkpoint(latest_ckpt)
else:
    print(f"==================================================")
    print(f"No checkpoint found under checkpoints/. Initiating new training.")
    print(f"==================================================")

best_val_metrics = trainer.fit()

## 8. Post-Training Evaluation Exports

In [ ]:
# ==========================================
# SECTION 8: EVALUATION METRICS & CONFUSION PLOTS
# ==========================================
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, precision_recall_fscore_support, accuracy_score

print("Loading best parameters checkpoint...")
best_ckpt = dirs["checkpoints"] / "best_model.pt"
trainer.load_checkpoint(best_ckpt)

model.eval()
all_spec_labels = []
all_sev_labels = []
all_spec_preds = []
all_sev_preds = []
all_routing_probs = []

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch["input_ids"].to(trainer.device)
        attention_mask = batch["attention_mask"].to(trainer.device)
        labels_spec = batch["labels_specialist"]
        labels_sev = batch["labels_severity"]
        
        outputs = model(input_ids, attention_mask)
        
        spec_preds = outputs.specialist_logits.argmax(dim=-1).cpu().numpy()
        sev_preds = outputs.severity_logits.argmax(dim=-1).cpu().numpy()
        
        all_spec_labels.extend(labels_spec.numpy())
        all_sev_labels.extend(labels_sev.numpy())
        all_spec_preds.extend(spec_preds)
        all_sev_preds.extend(sev_preds)
        
        if model._last_routing_decision is not None:
            all_routing_probs.append(model._last_routing_decision.routing_probabilities.cpu().numpy())

spec_acc = accuracy_score(all_spec_labels, all_spec_preds)
spec_p, spec_r, spec_f1, _ = precision_recall_fscore_support(all_spec_labels, all_spec_preds, average="macro", zero_division=0)

sev_acc = accuracy_score(all_sev_labels, all_sev_preds)
sev_p, sev_r, sev_f1, _ = precision_recall_fscore_support(all_sev_labels, all_sev_preds, average="macro", zero_division=0)

metrics_export = {
    "specialist": {
        "accuracy": float(spec_acc),
        "macro_precision": float(spec_p),
        "macro_recall": float(spec_r),
        "macro_f1": float(spec_f1)
    },
    "severity": {
        "accuracy": float(sev_acc),
        "macro_precision": float(sev_p),
        "macro_recall": float(sev_r),
        "macro_f1": float(sev_f1)
    },
    "overall_losses": {
        "val_loss": float(best_val_metrics["val_loss"]),
        "val_specialist_loss": float(best_val_metrics["val_specialist_loss"]),
        "val_severity_loss": float(best_val_metrics["val_severity_loss"]),
        "val_cons_loss": float(best_val_metrics["val_cons_loss"]),
        "val_div_loss": float(best_val_metrics["val_div_loss"]),
        "val_ortho_loss": float(best_val_metrics["val_ortho_loss"])
    }
}

with open(dirs["exports"] / "metrics.json", "w", encoding="utf-8") as f:
    json.dump(metrics_export, f, indent=4)

if all_routing_probs:
    all_probs = np.concatenate(all_routing_probs, axis=0)
    B_t, M_t, N_t = all_probs.shape
    epsilon = 1e-9
    entropies = -np.sum(all_probs * np.log(all_probs + epsilon), axis=-1)
    util_argmax = all_probs.argmax(axis=-1)
    utilization_counts = [np.bincount(util_argmax[:, s], minlength=N_t).tolist() for s in range(M_t)]
    
    routing_export = {
        "mean_routing_entropy": float(entropies.mean()),
        "entropy_per_step": entropies.mean(axis=0).tolist(),
        "ctb_utilizations_per_step": utilization_counts,
        "average_reasoning_depth": M_t,
        "mean_confidence": float(np.max(all_probs, axis=-1).mean())
    }
    with open(dirs["exports"] / "routing_statistics.json", "w", encoding="utf-8") as f:
        json.dump(routing_export, f, indent=4)

history_df = pd.DataFrame(trainer.history)
history_df.to_csv(dirs["exports"] / "training_history.csv", index=False)
val_cols = [c for c in history_df.columns if "val_" in c or c in ["epoch", "time"]]
history_df[val_cols].to_csv(dirs["exports"] / "validation_history.csv", index=False)

plt.figure(figsize=(10, 5))
plt.plot(history_df["epoch"], history_df["train_loss"], label="Train Loss", marker="o")
plt.plot(history_df["epoch"], history_df["val_loss"], label="Val Loss", marker="x")
plt.title("E-PATH-CO-REASON Loss Curves")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.grid(True)
plt.legend()
plt.savefig(dirs["figures"] / "loss_curves.png")
plt.close()

plt.figure(figsize=(10, 5))
plt.plot(history_df["epoch"], history_df["train_specialist_acc"], label="Train Spec Acc", marker="o")
plt.plot(history_df["epoch"], history_df["val_specialist_acc"], label="Val Spec Acc", marker="x")
plt.plot(history_df["epoch"], history_df["train_severity_acc"], label="Train Sev Acc", marker="s")
plt.plot(history_df["epoch"], history_df["val_severity_acc"], label="Val Sev Acc", marker="d")
plt.title("E-PATH-CO-REASON Accuracy Curves")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.grid(True)
plt.legend()
plt.savefig(dirs["figures"] / "accuracy_curves.png")
plt.close()

plt.figure(figsize=(10, 8))
sns.heatmap(confusion_matrix(all_spec_labels, all_spec_preds), annot=True, fmt="d", cmap="Blues",
            xticklabels=validator.specialist_classes, yticklabels=validator.specialist_classes)
plt.title("Specialist Confusion Matrix")
plt.savefig(dirs["figures"] / "specialist_confusion_matrix.png")
plt.close()

plt.figure(figsize=(8, 6))
sns.heatmap(confusion_matrix(all_sev_labels, all_sev_preds), annot=True, fmt="d", cmap="Oranges",
            xticklabels=validator.severity_labels, yticklabels=validator.severity_labels)
plt.title("Severity Confusion Matrix")
plt.savefig(dirs["figures"] / "severity_confusion_matrix.png")
plt.close()

shutil.copyfile(dirs["checkpoints"] / "best_model.pt", exp_base / "best_model.pt")
shutil.copyfile(dirs["checkpoints"] / "latest_model.pt", exp_base / "latest_model.pt")

print("Post-training assets successfully compiled and exported.")

## 9. Final Experiment Summary Report

In [ ]:
# ==========================================
# SECTION 9: FINAL EXPERIMENT SUMMARY
# ==========================================
print(f"==================================================")
print(f"FINAL EXPERIMENT SUMMARY REPORT")
print(f"==================================================")
print(f"Experiment Name      : {EXPERIMENT_CONFIG['experiment_name']}")
print(f"GPU Model Used       : {env_info['gpu_model']}")
print(f"Best Training Epoch  : {best_val_metrics.get('epoch', 'N/A')}")
print(f"Specialist Accuracy  : {spec_acc:.2%}")
print(f"Severity Accuracy    : {sev_acc:.2%}")
print(f"Specialist F1-Score  : {spec_f1:.2%}")
print(f"Severity F1-Score    : {sev_f1:.2%}")
print(f"--------------------------------------------------")
print(f"Outputs Directory Locations:")
print(f"- Checkpoints        : {dirs['checkpoints']}")
print(f"- Exported Reports   : {dirs['exports']}")
print(f"- Plot Figures       : {dirs['figures']}")
print(f"==================================================")